# Tutorial: TCGA ETL - Building Flat Slide Tables with Manifest Generation

This notebook demonstrates the complete ETL pipeline:
1. **Configure** - Set up project selection and directories
2. **Build table** - Query GDC API and build flat DataFrame
3. **Add local paths** - Compute where files will be downloaded
4. **Generate manifests** - Create manifest files for gdc-client
5. **Download** - Use gdc-client to download files

**Architecture:**
```
TCGAConfig → TCGASlideETL → ManifestGenerator → TCGADownloader
```

Each module has ONE job:
- `TCGAConfig` - Configuration management
- `TCGASlideETL` - Data transformation (queries → DataFrame)
- `ManifestGenerator` - Manifest file generation
- `TCGADownloader` - Download orchestration

## 1. Setup

In [1]:
import sys
sys.path.insert(0, '../../../..')  # Add project root to path

from pathlib import Path
import pandas as pd

from src.data.tcga import (
    TCGAConfig,
    TCGASlideETL,
    ManifestGenerator,
    TCGADownloader,
    DownloadStatus,
)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Setup complete!")

Setup complete!


## 2. Configure the Pipeline

`TCGAConfig` holds all settings. You must specify `project_ids`.

In [2]:
# Create configuration
config = TCGAConfig(
    project_ids=["TCGA-LUAD"],  # Lung Adenocarcinoma
    data_dir=Path("data/tcga"),
    include_demographics=True,
    include_diagnosis=True,
    include_maf=True,
    access="open",
)

print("Configuration:")
print(f"  project_ids: {config.project_ids}")
print(f"  data_dir: {config.data_dir}")
print(f"  slides_dir: {config.slides_dir}")
print(f"  maf_dir: {config.maf_dir}")
print(f"  manifests_dir: {config.manifests_dir}")
print(f"  tables_dir: {config.tables_dir}")

Configuration:
  project_ids: ['TCGA-LUAD']
  data_dir: data/tcga
  slides_dir: data/tcga/slides
  maf_dir: data/tcga/maf
  manifests_dir: data/tcga/manifests
  tables_dir: data/tcga/tables


In [3]:
# Create directories
config.ensure_directories()
print("Directories created!")

Directories created!


## 3. Build the Flat Slide Table

This queries the GDC API and builds a DataFrame with:
- File metadata (file_id, filename, file_size, **md5sum**)
- Slide data (slide_id, percent_tumor_cells)
- Sample data broadcast (sample_id, sample_type)
- Case data broadcast (demographics, diagnosis)
- MAF data at sample level (**full metadata** for manifest generation)

In [4]:
etl = TCGASlideETL()

print("Building slide table (fetches data from GDC API)...")
df = etl.build_slide_table(
    project_ids=config.project_ids,
    include_demographics=config.include_demographics,
    include_diagnosis=config.include_diagnosis,
    include_maf=config.include_maf,
    access=config.access,
)

print(f"\nTable shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Building slide table (fetches data from GDC API)...

Table shape: (1608, 34)
Columns: ['file_id', 'filename', 'file_size', 'md5sum', 'file_state', 'project_id', 'slide_id', 'slide_submitter_id', 'percent_tumor_cells', 'percent_necrosis', 'portion_id', 'is_ffpe', 'sample_id', 'sample_submitter_id', 'sample_type', 'tissue_type', 'case_id', 'case_submitter_id', 'gender', 'race', 'ethnicity', 'year_of_birth', 'primary_diagnosis', 'diagnosis_is_primary', 'tumor_stage', 'tumor_grade', 'vital_status', 'days_to_death', 'age_at_diagnosis', 'maf_file_id', 'maf_filename', 'maf_file_size', 'maf_md5sum', 'has_maf']


In [5]:
# Show all columns with their data types
print("COLUMN DETAILS:")
print("=" * 60)
for col in df.columns:
    non_null = df[col].notna().sum()
    dtype = df[col].dtype
    print(f"  {col:<25} | {dtype:<10} | {non_null}/{len(df)} non-null")

COLUMN DETAILS:


TypeError: unsupported format string passed to StringDtype.__format__

In [6]:
# Show first row transposed (easier to read)
print("FIRST ROW (transposed):")
print("=" * 60)
df.head(1).T

FIRST ROW (transposed):


,0
file_id,6a0ea716-a5f2-47f3-880b-537a5cdc2324
filename,TCGA-86-8074-01Z-00-DX1.0c34b434-8701-4060-a4e...
file_size,532458405
md5sum,0b2aa43f79e85d4ffe629418483e26c5
file_state,released
project_id,TCGA-LUAD
slide_id,4f65d8d8-a4b3-451f-b497-5e489f46ada1
slide_submitter_id,TCGA-86-8074-01Z-00-DX1
percent_tumor_cells,NaN
percent_necrosis,NaN


### Verify New Fields

The ETL now includes:
- `md5sum` - for slide files (needed for manifest)
- `maf_filename`, `maf_file_size`, `maf_md5sum` - full MAF metadata

In [5]:
print("SLIDE FILE METADATA:")
print("=" * 60)
print(f"md5sum present: {'md5sum' in df.columns}")
print(f"md5sum sample: {df['md5sum'].iloc[0]}")
print(f"md5sum null count: {df['md5sum'].isna().sum()}")

SLIDE FILE METADATA:
md5sum present: True
md5sum sample: 0b2aa43f79e85d4ffe629418483e26c5
md5sum null count: 0


In [6]:
print("MAF FILE METADATA:")
print("=" * 60)

# Show a row with MAF
maf_rows = df[df['has_maf']]
print(f"Slides with MAF: {len(maf_rows)} ({len(maf_rows)/len(df)*100:.1f}%)")

if not maf_rows.empty:
    row = maf_rows.iloc[0]
    print(f"\nSample MAF data:")
    print(f"  maf_file_id: {row['maf_file_id']}")
    print(f"  maf_filename: {row['maf_filename']}")
    print(f"  maf_file_size: {row['maf_file_size']}")
    print(f"  maf_md5sum: {row['maf_md5sum']}")

MAF FILE METADATA:
Slides with MAF: 934 (58.1%)

Sample MAF data:
  maf_file_id: 61300c06-3272-45e4-b19f-94dfd276debd
  maf_filename: f3934f06-2c19-40b7-88dc-b46d1a573bdc.wxs.aliquot_ensemble_masked.maf.gz
  maf_file_size: 29232.0
  maf_md5sum: 5153eb8081c86b3e7fe5f19f904d1149


## 4. Add Local File Paths

Compute where files will be downloaded. gdc-client creates:
```
<dir>/<file_uuid>/<filename>
```

In [7]:
# Add local path columns
df = etl.add_local_paths(df, config)

print("LOCAL PATHS ADDED:")
print("=" * 60)
print(f"slide_local_path: {df['slide_local_path'].iloc[0]}")

if not maf_rows.empty:
    maf_path = df[df['has_maf']]['maf_local_path'].iloc[0]
    print(f"maf_local_path: {maf_path}")

LOCAL PATHS ADDED:
slide_local_path: data/tcga/slides/6a0ea716-a5f2-47f3-880b-537a5cdc2324/TCGA-86-8074-01Z-00-DX1.0c34b434-8701-4060-a4ea-08a72371ee1e.svs
maf_local_path: data/tcga/maf/61300c06-3272-45e4-b19f-94dfd276debd/f3934f06-2c19-40b7-88dc-b46d1a573bdc.wxs.aliquot_ensemble_masked.maf.gz


## 5. Generate Manifests

`ManifestGenerator` creates manifest files for gdc-client.

Format: `id\tfilename\tmd5\tsize\tstate`

In [8]:
manifest_gen = ManifestGenerator()

# Create slide manifest
slide_manifest = manifest_gen.create_slide_manifest(
    df, config.manifests_dir / "slides_manifest.txt"
)
print(f"Slide manifest: {slide_manifest}")
print(f"  Files: {len(df)}")

# Create MAF manifest (deduplicated - many slides share same MAF)
maf_manifest = manifest_gen.create_maf_manifest(
    df, config.manifests_dir / "maf_manifest.txt"
)
if maf_manifest:
    unique_maf = df[df['has_maf']]['maf_file_id'].nunique()
    print(f"\nMAF manifest: {maf_manifest}")
    print(f"  Files: {unique_maf} (deduplicated)")

Slide manifest: data/tcga/manifests/slides_manifest.txt
  Files: 1608

MAF manifest: data/tcga/manifests/maf_manifest.txt
  Files: 510 (deduplicated)


In [9]:
# Show manifest content
print("SLIDE MANIFEST (first 5 lines):")
print("=" * 60)
with open(slide_manifest) as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(line.strip()[:100] + "..." if len(line) > 100 else line.strip())

SLIDE MANIFEST (first 5 lines):
id	filename	md5	size	state
6a0ea716-a5f2-47f3-880b-537a5cdc2324	TCGA-86-8074-01Z-00-DX1.0c34b434-8701-4060-a4ea-08a72371ee1e.sv...
b059ce82-63d7-43b4-b52b-a681daeaef5a	TCGA-86-8074-01A-01-BS1.9b6a32f7-07bb-4e59-aa64-487c6a43edab.sv...
c5cc9280-d4c9-4090-9fb1-1d1cc5e2f8d6	TCGA-78-7535-01Z-00-DX1.c4ca06f3-22d1-4e39-85c8-98d1fa2b0e60.sv...
22a6cf23-604a-4163-aa84-415fa93d6f58	TCGA-38-7271-01A-01-TS1.8e38324c-8605-4f16-a842-082e4ffd94dd.sv...


## 6. Download Files

`TCGADownloader` orchestrates downloads using gdc-client.

**Requirements:**
- gdc-client must be installed: `pip install gdc-client`
- For controlled access data: provide a token file

**Resume:** gdc-client automatically resumes interrupted downloads.

In [10]:
# Check download status (before downloading)
downloader = TCGADownloader()

status = downloader.check_download_status(
    output_dir=config.slides_dir,
    manifest_path=slide_manifest,
)

print("DOWNLOAD STATUS:")
print("=" * 60)
print(f"Status: {status.status.value}")
print(f"Files total: {status.files_total}")
print(f"Files downloaded: {status.files_downloaded}")

DOWNLOAD STATUS:
Status: in_progress
Files total: 1608
Files downloaded: 2


In [ ]:
# Create a SUBSET manifest for testing (first 2 files only)
test_manifest = manifest_gen.create_subset_manifest(
    manifest_path=slide_manifest,
    output_path=config.manifests_dir / "slides_test_manifest.txt",
    max_files=2,
)
print(f"Test manifest created: {test_manifest}")
print(f"  Files: 2 (subset for testing)")

# Show the test manifest
with open(test_manifest) as f:
    print(f"\nTest manifest contents:")
    print(f.read())

# Download just the test subset
result = downloader.download_from_manifest(
    manifest_path=test_manifest,  # Use test manifest, not full
    output_dir=config.slides_dir,
    n_processes=4,
)
print(f"Status: {result.status.value}")
print(f"Downloaded: {result.files_downloaded}/{result.files_total}")

print("\nDownload commands (run in terminal):")
print(f"  # Test (2 files):  gdc-client download -m {test_manifest} -d {config.slides_dir}")
print(f"  # Full (all files): gdc-client download -m {slide_manifest} -d {config.slides_dir}")

Test manifest created: data/tcga/manifests/slides_test_manifest.txt
  Files: 2 (subset for testing)

Test manifest contents:
id	filename	md5	size	state
6a0ea716-a5f2-47f3-880b-537a5cdc2324	TCGA-86-8074-01Z-00-DX1.0c34b434-8701-4060-a4ea-08a72371ee1e.svs	0b2aa43f79e85d4ffe629418483e26c5	532458405	released
b059ce82-63d7-43b4-b52b-a681daeaef5a	TCGA-86-8074-01A-01-BS1.9b6a32f7-07bb-4e59-aa64-487c6a43edab.svs	ed38614e8a75b7186deb161044ab2edc	1667746421	released

Status: completed
Downloaded: 2/2

Download commands (run in terminal):
  # Test (2 files):  gdc-client download -m data/tcga/manifests/slides_test_manifest.txt -d data/tcga/slides
  # Full (all files): gdc-client download -m data/tcga/manifests/slides_manifest.txt -d data/tcga/slides


In [17]:
# Create a SUBSET MAF manifest for testing (first 2 files only)
if maf_manifest:
    test_maf_manifest = manifest_gen.create_subset_manifest(
        manifest_path=maf_manifest,
        output_path=config.manifests_dir / "maf_test_manifest.txt",
        max_files=2,
    )
    print(f"Test MAF manifest created: {test_maf_manifest}")
    print(f"  Files: 2 (subset for testing)")

    # Show the test MAF manifest
    with open(test_maf_manifest) as f:
        print(f"\nTest MAF manifest contents:")
        print(f.read())

    # Download just the test subset
    result = downloader.download_from_manifest(
        manifest_path=test_maf_manifest,  # Use test manifest, not full
        output_dir=config.maf_dir,
        n_processes=4,
    )
    print(f"Status: {result.status.value}")
    print(f"Downloaded: {result.files_downloaded}/{result.files_total}")

    print("\nMAF Download commands (run in terminal):")
    print(f"  # Test (2 files):  gdc-client download -m {test_maf_manifest} -d {config.maf_dir}")
    print(f"  # Full (all files): gdc-client download -m {maf_manifest} -d {config.maf_dir}")

Test MAF manifest created: data/tcga/manifests/maf_test_manifest.txt
  Files: 2 (subset for testing)

Test MAF manifest contents:
id	filename	md5	size	state
61300c06-3272-45e4-b19f-94dfd276debd	f3934f06-2c19-40b7-88dc-b46d1a573bdc.wxs.aliquot_ensemble_masked.maf.gz	5153eb8081c86b3e7fe5f19f904d1149	29232.0	
2adbac87-1656-4f60-b224-b10feef293ec	e65fc432-29d8-46b0-b083-84b673f70630.wxs.aliquot_ensemble_masked.maf.gz	00a58d72936859473c5a247d226284c0	20158.0	

Status: completed
Downloaded: 2/2

MAF Download commands (run in terminal):
  # Test (2 files):  gdc-client download -m data/tcga/manifests/maf_test_manifest.txt -d data/tcga/maf
  # Full (all files): gdc-client download -m data/tcga/manifests/maf_manifest.txt -d data/tcga/maf


## 7. Save the Table

In [ ]:
# Save to CSV
output_path = config.tables_dir / "slides_table.csv"
df.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Shape: {df.shape}")

## 8. Data Exploration

In [ ]:
print("PRIMARY DIAGNOSIS:")
print(df['primary_diagnosis'].value_counts())

In [ ]:
print("SAMPLE TYPE:")
print(df['sample_type'].value_counts())

In [ ]:
print("MAF COVERAGE BY SAMPLE TYPE:")
print("(MAF files are for tumor mutations - normal tissue doesn't have MAF)")
print()
maf_coverage = df.groupby('sample_type')['has_maf'].agg(['sum', 'count', 'mean'])
maf_coverage.columns = ['with_maf', 'total', 'rate']
print(maf_coverage)

In [ ]:
print("FILE SIZE STATISTICS:")
print(f"Min: {df['file_size'].min() / 1e6:.1f} MB")
print(f"Max: {df['file_size'].max() / 1e9:.2f} GB")
print(f"Mean: {df['file_size'].mean() / 1e6:.1f} MB")
print(f"Total: {df['file_size'].sum() / 1e12:.2f} TB")

## 9. Multi-Project Example

In [ ]:
# Example: Both lung cancer types
config_lung = TCGAConfig(
    project_ids=["TCGA-LUAD", "TCGA-LUSC"],
    data_dir=Path("data/tcga_lung"),
)

print(f"Multi-project config: {config_lung.project_ids}")

# To build table:
# df_lung = etl.build_slide_table(
#     project_ids=config_lung.project_ids,
#     include_maf=True,
# )

## Summary

### Pipeline
```
1. TCGAConfig      → Configure projects and directories
2. TCGASlideETL    → Build flat DataFrame from GDC API
3. add_local_paths → Compute download paths
4. ManifestGenerator → Create manifest files
5. TCGADownloader  → Download files (or use gdc-client directly)
6. GeneMatrix      → Build gene mutation matrix from MAF files
```

### Key Classes

| Class | Responsibility |
|-------|---------------|
| `TCGAConfig` | Configuration management |
| `TCGASlideETL` | Query GDC API → flat DataFrame |
| `ManifestGenerator` | Create manifest files for gdc-client |
| `TCGADownloader` | Download orchestration |
| `GeneMatrix` | Gene-level one-hot encoding from MAF files |

### GeneMatrix Usage
```python
# Build from downloaded MAF files
gm = GeneMatrix()
gm.build_from_maf_dir(config.maf_dir)
gm.save(config.tables_dir / "gene_matrix.parquet")

# Load and merge with slides
gm = GeneMatrix.load(config.tables_dir / "gene_matrix.parquet")
df_with_genes = gm.merge(slide_df, genes=["TP53", "KRAS", "EGFR"])
```

### Notes
- MAF files link at **sample level** - all slides from same sample share mutations
- Normal tissue samples don't have MAF (MAF is for tumor mutations)
- GeneMatrix joins on `sample_id` (UUID) - slides without MAF get 0 for all genes
- gdc-client handles resume automatically

## 10. Gene Matrix - One-Hot Encoding from MAF Files

After downloading MAF files, `GeneMatrix` extracts gene-level mutation data.

**The challenge:** MAF files use aliquot UUIDs (`Tumor_Sample_UUID`), not sample UUIDs. We need to resolve this via the GDC API.

**TCGA hierarchy:**
```
Case → Sample → Portion → Analyte → Aliquot
```

**What GeneMatrix does:**
1. Parse MAF files → extract aliquot UUIDs and mutated genes
2. Query GDC API → resolve aliquot → sample mapping
3. Build matrix → rows = samples, columns = genes, values = 0/1
4. Merge with slide table → joins on sample_id

In [11]:
# Import GeneMatrix
# GeneMatrix reads MAF files and creates a gene-level one-hot matrix
# It uses the GDC API to resolve aliquot UUIDs → sample UUIDs

from src.data.tcga import GeneMatrix, GDCClient

# Find MAF files in the download directory
# gdc-client downloads to: maf_dir/<file_uuid>/<filename>.maf.gz
maf_files = list(config.maf_dir.glob("*/*.maf.gz"))
print(f"Found {len(maf_files)} MAF files:")
for f in maf_files:
    print(f"  - {f.name}")

Found 2 MAF files:
  - f3934f06-2c19-40b7-88dc-b46d1a573bdc.wxs.aliquot_ensemble_masked.maf.gz
  - e65fc432-29d8-46b0-b083-84b673f70630.wxs.aliquot_ensemble_masked.maf.gz


In [12]:
# Peek inside a MAF file to understand its structure
# MAF = Mutation Annotation Format - one row per mutation
# 
# Key columns:
# - Hugo_Symbol: gene name (e.g., TP53, KRAS)
# - Tumor_Sample_UUID: this is actually an ALIQUOT UUID (not sample!)
# - Variant_Classification: type of mutation
#
# TCGA hierarchy: Case → Sample → Portion → Analyte → Aliquot
# So we need to resolve aliquot → sample via GDC API

if maf_files:
    maf_sample = pd.read_csv(maf_files[0], sep='\t', comment='#', nrows=5)
    
    print(f"MAF file has {len(maf_sample.columns)} columns")
    print(f"\nKey columns:")
    
    key_cols = ['Hugo_Symbol', 'Variant_Classification', 'Tumor_Sample_UUID']
    print(maf_sample[key_cols])

MAF file has 140 columns

Key columns:
  Hugo_Symbol Variant_Classification                     Tumor_Sample_UUID
0    PRAMEF15      Missense_Mutation  699460ce-e908-4d17-8425-b7fca3bfbeb9
1       KTI12      Missense_Mutation  699460ce-e908-4d17-8425-b7fca3bfbeb9
2         FLG                 Silent  699460ce-e908-4d17-8425-b7fca3bfbeb9
3     ISG20L2                 Silent  699460ce-e908-4d17-8425-b7fca3bfbeb9
4       INSRR      Missense_Mutation  699460ce-e908-4d17-8425-b7fca3bfbeb9


In [13]:
# Build the gene mutation matrix from all MAF files
# This does 3 things:
# 1. Parse MAF files to extract aliquot UUIDs and mutated genes
# 2. Query GDC API to resolve aliquot → sample mapping
# 3. Build matrix indexed by sample_id (so it can join with slide table)

client = GDCClient()
gm = GeneMatrix(client=client)
gm.build_from_maf_dir(config.maf_dir)

print(f"Built: {gm}")
print(f"  - {len(gm.samples)} samples")
print(f"  - {len(gm.genes)} unique mutated genes")

Built: GeneMatrix(samples=2, genes=125)
  - 2 samples
  - 125 unique mutated genes


In [17]:
# View the gene matrix - each cell is 1 (gene mutated) or 0 (not mutated)
# With only 2 MAF files, we have 2 samples and their mutated genes

matrix_df = gm.to_dataframe()
print(f"Matrix shape: {matrix_df.shape}")
print(f"\nFirst 10 genes:")
matrix_df.iloc[:, :10]

Matrix shape: (2, 125)

First 10 genes:


,ADAMTSL3,ADCY1,ADCY8,ADM,APOB,ARFGEF3,ARPP21,BBC3,BIRC6,BUB1
sample_id,,,,,,,,,,
0382e297-1b96-433b-9440-6811d685b13b,1,0,1,0,1,1,1,1,0,1
ed2389b5-16eb-43e8-87b8-a15304da629a,0,1,0,1,0,0,0,0,1,0


In [18]:
# Subset to specific genes of interest
# Common cancer driver genes - may or may not be mutated in our 2-sample test set
# Genes not in the matrix get filled with 0

cancer_genes = ["TP53", "KRAS", "EGFR", "BRAF", "PIK3CA"]
subset = gm.subset(cancer_genes)

print(f"Subsetting to {len(cancer_genes)} genes:")
subset

Subsetting to 5 genes:


,TP53,KRAS,EGFR,BRAF,PIK3CA
sample_id,,,,,
0382e297-1b96-433b-9440-6811d685b13b,1,0,1,0,0
ed2389b5-16eb-43e8-87b8-a15304da629a,0,0,0,0,0


In [19]:
# Before merging, verify sample_id linkage
# Gene matrix uses Tumor_Sample_UUID which should match sample_id in slide table

print("Sample IDs in gene matrix:")
for sid in gm.samples:
    print(f"  {sid}")


# Find matching slides
matching_slides = df[df['sample_id'].isin(gm.samples)]
print(f"\nSlides that match gene matrix samples: {len(matching_slides)}")

Sample IDs in gene matrix:
  0382e297-1b96-433b-9440-6811d685b13b
  ed2389b5-16eb-43e8-87b8-a15304da629a

Slides that match gene matrix samples: 3


In [20]:
# Merge gene mutations into the slide table
# This joins on sample_id - all slides from the same sample get the same gene values
# Slides without MAF data get 0 for all genes (left join)

merged_df = gm.merge(df, genes=cancer_genes)

print(f"Original: {df.shape[1]} columns")
print(f"After merge: {merged_df.shape[1]} columns")
print(f"Added columns: {cancer_genes}")

Original: 36 columns
After merge: 41 columns
Added columns: ['TP53', 'KRAS', 'EGFR', 'BRAF', 'PIK3CA']


In [21]:
# View slides that have gene mutation data
# These are slides whose sample_id matched the gene matrix

cols = ['filename', 'sample_id'] + cancer_genes
matched = merged_df[merged_df['sample_id'].isin(gm.samples)]

print(f"Slides with gene data: {len(matched)}")
matched[cols].head()

Slides with gene data: 3


,filename,sample_id,TP53,KRAS,EGFR,BRAF,PIK3CA
1,TCGA-86-8074-01A-01-BS1.9b6a32f7-07bb-4e59-aa6...,0382e297-1b96-433b-9440-6811d685b13b,1,0,1,0,0
3,TCGA-38-7271-01A-01-TS1.8e38324c-8605-4f16-a84...,ed2389b5-16eb-43e8-87b8-a15304da629a,0,0,0,0,0
1201,TCGA-86-8074-01A-01-TS1.529fc24b-76e2-422c-a4a...,0382e297-1b96-433b-9440-6811d685b13b,1,0,1,0,0


In [22]:
# View slides WITHOUT gene data (most slides in our test)
# Left join preserved them with 0s for all gene columns

unmatched = merged_df[~merged_df['sample_id'].isin(gm.samples)]

print(f"Slides without gene data: {len(unmatched)}")
print("(Gene columns are all 0 for these)")
unmatched[cols].head()

Slides without gene data: 1605
(Gene columns are all 0 for these)


,filename,sample_id,TP53,KRAS,EGFR,BRAF,PIK3CA
0,TCGA-86-8074-01Z-00-DX1.0c34b434-8701-4060-a4e...,eed595f6-cb7e-445b-9a27-85a6f353ca96,0,0,0,0,0
2,TCGA-78-7535-01Z-00-DX1.c4ca06f3-22d1-4e39-85c...,16a45616-5855-4ed8-9e17-3993823e4360,0,0,0,0,0
4,TCGA-86-8358-01Z-00-DX1.C50100F8-9414-4A06-BED...,41e6e10e-491f-4ed1-a07f-865566fe5f93,0,0,0,0,0
5,TCGA-86-8358-01A-01-BS1.52e2fcc8-cbc7-4239-ae6...,5942f197-2b77-4a7f-8bca-763cc1902a0b,0,0,0,0,0
6,TCGA-78-7158-01A-01-BS1.3b9b6712-5409-45b1-8f2...,ee0d4640-5609-430e-8b80-3058a3f313a8,0,0,0,0,0


In [ ]:
# Save the gene matrix for later use
# Parquet format is efficient for sparse data like this

gene_matrix_path = config.tables_dir / "gene_matrix.parquet"
gm.save(gene_matrix_path)
print(f"Saved to: {gene_matrix_path}")

# Load it back to verify
gm_loaded = GeneMatrix.load(gene_matrix_path)
print(f"Loaded: {gm_loaded}")